In [ ]:
from general_utils import find_ephys_sessions

sessions=find_ephys_sessions()

In [ ]:
import numpy as np
from ephys_dimension_reduction_tdr import tdr_from_psth
from create_psth import load_zarr
from general_utils import smart_read_csv

failed_sessions = []

for session in sessions[2]:
    try:
        binsize = '0.1'
        time_window=[-1,0]
        align = "go_cue"
        #latent_var = 'ForagingCompareThreshold-deltaQ-1'
        latent_var = 'QLearning_L2F1_softmax-deltaQ-1'
        psth_da = load_zarr(f"/root/capsule/scratch/{session}_{binsize}s.zarr")
        df = smart_read_csv(f"/root/capsule/data/regression_result/behavior_summary-{session}.csv")
        #latent_var = 'QLearning_L2F1_softmax-sumQ-1'
        keep_ids = np.asarray(df['response_trials'][0], dtype=int)
        latent_full = np.asarray(df[latent_var][0], dtype=float)

        out = tdr_from_psth(
            psth_da,
            latent=latent_full,
            align=align,
            time_window=time_window,
            include_trials=keep_ids,
            require_all_ids=True,
            save_path=f"/root/capsule/scratch/tdr_{session}_{latent_var}_timewindow_{time_window[0]}_{time_window[1]}.zarr",
            save_format="zarr",
        )

    except Exception as e:
        print(f"❌ Error in session {session}: {e}")
        failed_sessions.append(session)

print("\nAll sessions done.")
if failed_sessions:
    print("Failed sessions:")
    for s in failed_sessions:
        print(f" - {s}")
else:
    print("No errors 🎉")
